In [1]:
!py -m pip install openai python-dotenv langchain langchain-openai langgraph pydantic --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import required libraries
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Setup complete! OpenAI client initialized.")

Setup complete! OpenAI client initialized.


In [21]:
import fs_tools;
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a file and return the content in a structured format with file name, extension, size, and content",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The path to the file to read"
                    }
                },
                "required": ["filepath"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files in a directory and return the results in a structured format with file names, sizes, and modification times",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {
                        "type": "string",
                        "description": "The path to the directory to list"
                    },
                    "extension": {
                        "type": "string",
                        "description": "The extension of the files to list"
                    }
                },
                "required": ["directory", "extension"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write content to a file and return the status of the operation in a structured format",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The path to the file to write to in the current directory"
                    },
                    "content": {
                        "type": "string",
                        "description": "The content to write to the file in the current directory"
                    }
                },
                "required": ["filepath", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_in_file",
            "description": "Search for a keyword in a file and return the results in a structured format with line numbers and context",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The path to the file to search in the current directory"
                    },
                    "keyword": {
                        "type": "string",
                        "description": "The keyword to search for in the current directory"
                    }
                },
                "required": ["filepath", "keyword"]
            }
        }
    }
]

TOOL_FUNCTIONS = {
    "read_file": fs_tools.read_file,
    "list_files": fs_tools.list_files,
    "write_file": fs_tools.write_file,
    "search_in_file": fs_tools.search_in_file
}

In [22]:
# The Tool-Using Agent
import json
class ToolAgent:

    def __init__(self):
        self.client = OpenAI()
        self.tools = TOOLS
        self.tool_functions = TOOL_FUNCTIONS
    
    def run(self, user_message: str, verbose: bool = True) -> str:
        messages = [
            {
                "role": "system",
                "content": """You are a helpful assistant with access to tools.
                Use tools when needed to provide accurate information.
                Always explain what you're doing and interpret tool results for the user."""
            },
            {"role": "user", "content": user_message}
        ]

        while True:
            # Get LLM response
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=self.tools,
                tool_choice="auto"
            )
            
            assistant_message = response.choices[0].message
            # Check if we need to call tools
            if assistant_message.tool_calls:
                messages.append(assistant_message)
                
                # Execute each tool call
                for tool_call in assistant_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)
                    
                    if verbose:
                        print(f"🔧 TOOL CALL: {function_name}({function_args})")
                    
                    # Execute the tool
                    if function_name in self.tool_functions:
                        result = self.tool_functions[function_name](**function_args)
                    else:
                        result = f"Unknown tool: {function_name}"
                    
                    result = json.dumps(result)
                    if verbose:
                        print(f"📋 RESULT: {result}\n")
                    
                    # Add tool result to messages
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result
                    })
            else:
                # No more tool calls, return final response
                return assistant_message.content

In [25]:
# Test the Tool-Using Agent
agent = ToolAgent()
test_queries = [
    'list all files in the current directory convert time to human readable format',
    'create a new file called "test.txt" in the current directory and write "Hello, world!" to it',
    'search for "hello" in all files in the current directory',
    "what is today's date?"
]

for query in test_queries:
    print("=" * 60)
    print(f"USER: {query}")
    print("-" * 60)
    response = agent.run(query)
    print(f"\nASSISTANT: {response}")
    print()

USER: list all files in the current directory convert time to human readable format
------------------------------------------------------------
🔧 TOOL CALL: list_files({'directory': './', 'extension': ''})
📋 RESULT: [{"name": "fs_tools.py", "path": "./fs_tools.py", "size_bytes": 4347, "modified_time": 1772712052.786654}, {"name": "LLM-Powered File System Assistant.ipynb", "path": "./LLM-Powered File System Assistant.ipynb", "size_bytes": 6788, "modified_time": 1772710618.4843507}, {"name": "test.txt", "path": "./test.txt", "size_bytes": 13, "modified_time": 1772712168.1113448}]


ASSISTANT: Here are the files in the current directory along with their sizes and human-readable modification times:

1. **File Name**: `fs_tools.py`
   - **Size**: 4,347 bytes
   - **Modified Time**: November 30, 2022, 10:30:52 AM

2. **File Name**: `LLM-Powered File System Assistant.ipynb`
   - **Size**: 6,788 bytes
   - **Modified Time**: November 30, 2022, 10:23:38 AM

3. **File Name**: `test.txt`
   - **